# NS10657 — reconcile the counter–sensor matching distance

For every retained row of the released processed dataset, compute the distance |dt| to the
nearest raw environmental sample, using a **global** sorted environmental time axis so that
spans without environmental coverage appear as long |dt| instead of being dropped.

Reconcile against the p2_checks **C3** numbers: median 0.41 s, p95 4.3 s, max 458 s, ~1.5% > 10 s.

- Run from anywhere: the release root is found automatically.
- Raw path (portable SSD) can be overridden with the environment variable `NS10657_RAW`.
- Read-only: nothing is written unless the last cell is run.

In [1]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd

# --- locate the release root (.../refractive_index_1762nm_v2.0) -----------------
here = Path.cwd()
root = next(p for p in [here, *here.parents]
            if (p / 'data' / 'processed' / 'full_data.csv').exists())
FULL = root / 'data' / 'processed' / 'full_data.csv'

RAW = Path(os.environ.get(
    'NS10657_RAW',
    '/Volumes/PortableSSD/Freiburg/Actomics/barium_network/zenodo/data/raw/'))

SENSORS = ['temperature']          # use ['temperature', 'humidity', 'pressure'] to check all three

print('release root :', root)
print('full_data    :', FULL)
print('raw          :', RAW, '| exists:', RAW.exists())
print('sensors      :', SENSORS)

release root : /Users/weiwu/Documents/科研相关/4_弗赖堡大学/TIAMO_personal/scripts/DataEvaluation/2025_07_ACTOMICS/zenodo/refractive_index_1762nm_v2.0
full_data    : /Users/weiwu/Documents/科研相关/4_弗赖堡大学/TIAMO_personal/scripts/DataEvaluation/2025_07_ACTOMICS/zenodo/refractive_index_1762nm_v2.0/data/processed/full_data.csv
raw          : /Volumes/PortableSSD/Freiburg/Actomics/barium_network/zenodo/data/raw | exists: True
sensors      : ['temperature']


In [2]:
df = pd.read_csv(FULL)
t = pd.to_datetime(df['time'], utc=True)
ct = t.values.astype('datetime64[ns]').astype('int64')      # counter times, ns
day = t.dt.strftime('%Y%m%d')
print(f'retained rows: {len(df)} over {day.nunique()} days')

summary = []
for sensor in SENSORS:
    parts, missing = [], []
    for d in sorted(day.unique()):
        f = RAW / 'environmental' / f'{sensor}_data_{d}.csv'
        if not f.exists():
            missing.append(d)
            continue
        e = pd.read_csv(f)
        et = pd.to_datetime(e['time'], utc=True, errors='coerce').dropna().values
        if et.size:
            parts.append(et.astype('datetime64[ns]').astype('int64'))

    env = np.sort(np.concatenate(parts)) if parts else np.array([], dtype='int64')
    idx = np.searchsorted(env, ct)
    best = np.full(ct.shape, np.iinfo(np.int64).max, dtype='int64')
    for k in (idx, np.clip(idx - 1, 0, env.size - 1)):
        best = np.minimum(best, np.abs(env[np.clip(k, 0, env.size - 1)] - ct))
    d = best / 1e9                                          # seconds

    row = dict(
        sensor=sensor, n=int(d.size), n_env_samples=int(env.size),
        median_s=float(np.median(d)), p95_s=float(np.percentile(d, 95)), max_s=float(d.max()),
        n_gt_10s=int((d > 10).sum()), frac_gt_10s=float((d > 10).mean()),
        n_gt_60s=int((d > 60).sum()), frac_gt_60s=float((d > 60).mean()),
        days_without_env_file=len(missing), days_without_env=missing,
    )
    summary.append(row)
    print(f"\n[{sensor}]  n={d.size}  median={row['median_s']:.3f} s  p95={row['p95_s']:.3f} s  "
          f"max={row['max_s']:.1f} s  |  >10 s: {row['n_gt_10s']} ({100*row['frac_gt_10s']:.2f}%)  "
          f">60 s: {row['n_gt_60s']} ({100*row['frac_gt_60s']:.3f}%)  "
          f"|  env samples: {env.size}  |  days without env file: {len(missing)}")

pd.DataFrame(summary)

retained rows: 145784 over 89 days

[temperature]  n=145784  median=0.405 s  p95=4.252 s  max=458.4 s  |  >10 s: 2114 (1.45%)  >60 s: 18 (0.012%)  |  env samples: 10700655  |  days without env file: 0


,sensor,n,n_env_samples,median_s,p95_s,max_s,n_gt_10s,frac_gt_10s,n_gt_60s,frac_gt_60s,days_without_env_file,days_without_env
0,temperature,145784,10700655,0.405151,4.252388,458.419577,2114,0.014501,18,0.000123,0,[]


### How to read the result

- If the `>10 s` fraction is **≈1.5%** with `max ≈458 s`, then **C3 is correct** and the F2
  notebook (`13_preprocessing_bound.ipynb`) is the one in error: its day-by-day merge skips
  days whose environmental file is missing, so those rows never receive a long |dt|.
- If the `>10 s` fraction is **0**, then the C3 numbers must be corrected in the SM.
- `days_without_env_file` (and the list) shows directly whether whole days lack environmental
  coverage; that is the suspected cause of the discrepancy.

In [3]:
# optional: save the reconciliation next to this notebook
out = Path('NS10657_match_reconcile.json')
out.write_text(json.dumps(summary, indent=2))
print('written:', out.resolve())

written: /Users/weiwu/Documents/科研相关/4_弗赖堡大学/TIAMO_personal/scripts/DataEvaluation/2025_07_ACTOMICS/zenodo/refractive_index_1762nm_v2.0/code/analysis/NS10657_match_reconcile.json
